# Model Querying Patterns

**Companion lesson:** https://ml-viz.vercel.app/courses/agent-design-patterns/05-model-querying-patterns

A from-scratch, runnable implementation of the concepts in the lesson — pure NumPy, no API keys required.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
BRAND = '#6366f1'; TEAL = '#2dd4bf'; ROSE = '#fb7185'; YELLOW = '#fbbf24'
np.random.seed(0)

## Mocked LLM

We simulate a foundation model with a deterministic mock. The mock inspects
keywords in the prompt and returns plausible-looking responses — no API key
needed. The querying *patterns* are identical regardless of whether the backend
is a mock or a real model.

In [ ]:
# ---------------------------------------------------------------------------
# Mocked LLM — keyword-dispatched responses that mimic a real model
# ---------------------------------------------------------------------------

_CALL_LOG = []   # global log so we can inspect token counts later

def _token_count(text: str) -> int:
    """Proxy: whitespace-split word count ≈ token count."""
    return len(text.split())

def mock_llm(prompt: str) -> str:
    """Simulated LLM call. Returns a canned response keyed on prompt content."""
    p = prompt.lower()
    if 'step 1' in p or 'first step' in p or 'start' in p:
        response = 'Step 1: Identify the two numbers to multiply. We have 17 and 24.'
    elif 'step 2' in p or 'partial product' in p:
        response = 'Step 2: Compute partial products. 17 × 20 = 340 and 17 × 4 = 68.'
    elif 'step 3' in p or 'add' in p or 'sum' in p:
        response = 'Step 3: Sum the partial products. 340 + 68 = 408.'
    elif 'final' in p or 'answer' in p or 'result' in p:
        response = 'The final answer is 408.'
    elif 'multiply' in p or 'product' in p or '17' in p:
        response = '408'
    else:
        response = 'I am not sure. Could you clarify?'

    in_tokens  = _token_count(prompt)
    out_tokens = _token_count(response)
    _CALL_LOG.append({'prompt_tokens': in_tokens, 'response_tokens': out_tokens})
    return response

# Sanity check
print(mock_llm('What is 17 multiplied by 24?'))

## One-shot querier

The `OneShotQuerier` packs *all* context and instruction into a single prompt
and calls the model exactly once. It is the simplest possible agent interface:
no state, no retry, one call, one answer.

In [ ]:
class OneShotQuerier:
    """Answers a task with a single LLM call."""

    def __init__(self, system_prompt: str = ''):
        self.system_prompt = system_prompt
        self.call_log: list[dict] = []

    def query(self, task: str) -> str:
        log_before = len(_CALL_LOG)
        prompt = f"{self.system_prompt}\n\nTask: {task}\n\nAnswer:" if self.system_prompt else f"Task: {task}\n\nAnswer:"
        response = mock_llm(prompt)
        self.call_log.extend(_CALL_LOG[log_before:])
        return response

    @property
    def total_tokens(self) -> int:
        return sum(e['prompt_tokens'] + e['response_tokens'] for e in self.call_log)

    @property
    def num_calls(self) -> int:
        return len(self.call_log)


system = 'You are a maths tutor. Answer concisely.'
task   = 'Compute 17 × 24. Just give the numeric result.'

_CALL_LOG.clear()
one_shot = OneShotQuerier(system_prompt=system)
answer   = one_shot.query(task)

print(f'Answer : {answer}')
print(f'Calls  : {one_shot.num_calls}')
print(f'Tokens : {one_shot.total_tokens}')

## Incremental querier

The `IncrementalQuerier` breaks the task into steps and issues a separate LLM
call for each. Each call receives the accumulated scratchpad from all prior
calls, so later steps can build on earlier reasoning.

We also add **error recovery**: if a step's response looks wrong (our mock
verifier checks for empty or error strings), the querier appends a correction
hint and re-issues that step — this is the loop-with-exit-condition pattern.

In [ ]:
class IncrementalQuerier:
    """
    Chains multiple LLM calls, threading a scratchpad between them.
    Supports per-step retry for error recovery.
    """

    def __init__(self, steps: list[str], max_retries: int = 1):
        """
        steps       : list of step-instruction strings
        max_retries : how many times to retry a step that fails verification
        """
        self.steps = steps
        self.max_retries = max_retries
        self.call_log: list[dict] = []
        self.scratchpad: list[str] = []

    @staticmethod
    def _verify(response: str) -> bool:
        """Simple verifier: response must be non-empty and not an error message."""
        bad_signals = ['not sure', 'clarify', 'error', 'unknown']
        return bool(response.strip()) and not any(s in response.lower() for s in bad_signals)

    def run(self, context: str) -> str:
        """Execute all steps, returning the final step's response."""
        self.scratchpad = [f'Context: {context}']
        log_before = len(_CALL_LOG)

        for i, step_instruction in enumerate(self.steps):
            for attempt in range(self.max_retries + 1):
                prompt = '\n'.join(self.scratchpad) + f'\n\nStep {i+1}: {step_instruction}'
                response = mock_llm(prompt)

                if self._verify(response):
                    self.scratchpad.append(f'Step {i+1} result: {response}')
                    break
                else:
                    # Recovery: append failure hint and retry
                    self.scratchpad.append(
                        f'Step {i+1} attempt {attempt+1} failed: "{response}". Retrying.'
                    )
            else:
                # All retries exhausted — record failure and continue
                self.scratchpad.append(f'Step {i+1}: FAILED after {self.max_retries+1} attempts.')

        self.call_log.extend(_CALL_LOG[log_before:])
        # Return the last successful scratchpad entry
        return self.scratchpad[-1]

    @property
    def total_tokens(self) -> int:
        return sum(e['prompt_tokens'] + e['response_tokens'] for e in self.call_log)

    @property
    def num_calls(self) -> int:
        return len(self.call_log)


cot_steps = [
    'Start: name the two numbers we are multiplying.',
    'Step 2: break 24 into 20 + 4 and compute the partial products.',
    'Step 3: add the partial products to get the final result.',
    'State the final answer.',
]

_CALL_LOG.clear()
incremental = IncrementalQuerier(steps=cot_steps, max_retries=1)
result = incremental.run('Compute 17 × 24.')

print('Scratchpad')
print('----------')
for line in incremental.scratchpad:
    print(' ', line)
print(f'\nCalls  : {incremental.num_calls}')
print(f'Tokens : {incremental.total_tokens}')

## Error recovery in action

To demonstrate the loop-with-exit-condition pattern, we inject a deliberately
ambiguous step that triggers the verifier to fail on the first attempt. The
querier appends the failure hint and re-issues the step — a simplified version
of what ReAct agents do when a tool call returns an error.

In [ ]:
# A step whose phrasing initially confuses the mock (no keywords it recognises)
recovery_steps = [
    'Please clarify what operation we need.',   # triggers 'clarify' → verifier fails
    'Now multiply 17 by 24 and give the answer.',
]

_CALL_LOG.clear()
recovery_q = IncrementalQuerier(steps=recovery_steps, max_retries=2)
recovery_q.run('We need to multiply two numbers.')

print('Scratchpad with recovery')
print('------------------------')
for line in recovery_q.scratchpad:
    print(' ', line)
print(f'\nCalls made (including retries): {recovery_q.num_calls}')

## Token and call counts

We now measure both approaches on the same arithmetic reasoning task and
compare total tokens, number of calls, and simulated latency.

**Simulated latency model:** each call has a fixed overhead of 200 ms plus
1 ms per output token (time-to-first-token is negligible here; we model
sequential generation time).

In [ ]:
MS_OVERHEAD  = 200   # ms per call (network + TTFT)
MS_PER_TOKEN = 1     # ms per output token (generation speed)

def simulate_latency(call_log: list[dict]) -> float:
    """Sum latency across sequential calls (they cannot overlap)."""
    total = 0.0
    for entry in call_log:
        total += MS_OVERHEAD + entry['response_tokens'] * MS_PER_TOKEN
    return total

# --- One-shot run ---
_CALL_LOG.clear()
os_q   = OneShotQuerier(system_prompt='You are a maths tutor.')
os_q.query('Compute 17 × 24 step by step, then give the numeric answer.')
os_log = list(_CALL_LOG)

# --- Incremental run ---
_CALL_LOG.clear()
inc_q  = IncrementalQuerier(steps=cot_steps, max_retries=1)
inc_q.run('Compute 17 × 24.')
inc_log = list(_CALL_LOG)

os_tokens   = sum(e['prompt_tokens'] + e['response_tokens'] for e in os_log)
inc_tokens  = sum(e['prompt_tokens'] + e['response_tokens'] for e in inc_log)
os_latency  = simulate_latency(os_log)
inc_latency = simulate_latency(inc_log)

print(f'              One-shot  Incremental')
print(f'Calls         {len(os_log):>8d}  {len(inc_log):>11d}')
print(f'Total tokens  {os_tokens:>8d}  {inc_tokens:>11d}')
print(f'Latency (ms)  {os_latency:>8.0f}  {inc_latency:>11.0f}')

## Visualisation — comparing the two patterns

Three side-by-side bar charts let us compare total tokens, number of calls,
and simulated latency at a glance. Notice that incremental querying uses more
tokens (the scratchpad grows with each call) and has higher latency (calls are
sequential), but enables richer reasoning and error recovery.

In [ ]:
labels   = ['One-shot', 'Incremental']
tokens   = [os_tokens,  inc_tokens]
calls    = [len(os_log), len(inc_log)]
latency  = [os_latency, inc_latency]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
bar_colors = [BRAND, TEAL]

for ax, values, title, ylabel in zip(
    axes,
    [tokens, calls, latency],
    ['Total tokens', 'Number of LLM calls', 'Simulated latency (ms)'],
    ['tokens', 'calls', 'ms'],
):
    bars = ax.bar(labels, values, color=bar_colors, width=0.5)
    ax.set_title(title, fontsize=11)
    ax.set_ylabel(ylabel)
    for bar, val in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + max(values) * 0.02,
            f'{val:.0f}',
            ha='center', va='bottom', fontsize=10, color='#e2e8f0'
        )
    ax.set_ylim(0, max(values) * 1.25)

plt.suptitle('One-shot vs Incremental querying — cost and latency comparison',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## ✏️ Your turn

**Challenge:** implement a `RetryQuerier` that wraps `IncrementalQuerier` and
retries an *entire run* (not just a single step) up to `max_retries` times if
the final answer does not pass a provided `success_fn` callable.

This is the outer retry loop pattern — used when you cannot know which step
failed, so you restart from scratch. It is common in agentic pipelines that
interact with external systems where partial state cannot be recovered.

```python
success_fn = lambda answer: '408' in answer
rq = RetryQuerier(steps=cot_steps, max_retries=3, success_fn=success_fn)
answer, attempts = rq.run('Compute 17 × 24.')
# answer should contain '408', attempts should be 1 for a mock that works first time
```

In [ ]:
class RetryQuerier:
    """
    Wraps IncrementalQuerier and retries the entire run up to max_retries times
    if success_fn(final_answer) returns False.

    Returns (final_answer: str, attempts_used: int).
    """

    def __init__(self, steps: list[str], max_retries: int, success_fn):
        # TODO(you): store steps, max_retries, and success_fn as instance attributes
        pass

    def run(self, context: str) -> tuple[str, int]:
        # TODO(you): loop up to max_retries+1 times.
        #   On each attempt, create a fresh IncrementalQuerier with self.steps
        #   and call .run(context). If success_fn(answer) is True, return
        #   (answer, attempt_number). After all retries, return the last answer.
        pass

In [ ]:
success_fn = lambda answer: '408' in answer
rq = RetryQuerier(steps=cot_steps, max_retries=3, success_fn=success_fn)
answer, attempts = rq.run('Compute 17 × 24.')

assert '408' in answer, f'Expected answer to contain 408, got: {answer}'
assert isinstance(attempts, int) and attempts >= 1, 'attempts should be a positive integer'
assert attempts <= 4, 'should not exceed max_retries + 1 attempts'
print('passed ✓')

<details><summary>Solution</summary>

```python
class RetryQuerier:
    def __init__(self, steps, max_retries, success_fn):
        self.steps = steps
        self.max_retries = max_retries
        self.success_fn = success_fn

    def run(self, context):
        last_answer = ''
        for attempt in range(1, self.max_retries + 2):  # +2: 1-indexed + extra slot
            q = IncrementalQuerier(steps=self.steps, max_retries=1)
            last_answer = q.run(context)
            if self.success_fn(last_answer):
                return last_answer, attempt
        return last_answer, self.max_retries + 1
```

</details>